<a href="https://colab.research.google.com/github/DhimanTarafdar/AAA/blob/main/ResNet50_notes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Block 1: Introduction to ResNet-50**

**ResNet-50** হলো একটি গভীর কনভোলিউশনাল নিউরাল নেটওয়ার্ক (Deep CNN) আর্কিটেকচার। এটি Microsoft Research দ্বারা ২০১৫ সালে প্রবর্তিত হয় এবং **ImageNet Large Scale Visual Recognition Challenge (ILSVRC)** এ চমৎকার ফলাফল অর্জন করে।

- **Full Form**: Residual Network with 50 layers.
- **Main Contribution**: **Vanishing Gradient Problem (গ্রেডিয়েন্ট বিলুপ্তি সমস্যা)** সমাধান করা।

VGG-16 এর মত নেটওয়ার্কের সমস্যা ছিল যে লেয়ার বাড়ানোর সাথে সাথে গ্রেডিয়েন্ট এতই ছোট হয়ে যেত যে আগের লেয়ারগুলো ঠিকমতো শিখতে পারত না। ResNet **"Skip Connection" (স্কিপ সংযোগ)** বা **"Identity Connection"** নামক কৌশল ব্যবহার করে এই সমস্যার সমাধান করে।

---


### **Block 2: Why ResNet?**

VGG নেটওয়ার্কের ১৬ বা ১৯ লেয়ার পার করার পর পারফরম্যান্স কমে যেতে শুরু করে। ResNet দেখায় যে, স্কিপ সংযোগ ব্যবহার করলে আমরা **১০০+ লেয়ার** পর্যন্ত নেটওয়ার্ক বাড়াতে পারি এবং ট্রেনিং ঠিকমতো হয়।

**VGG এর সীমাবদ্ধতা:**

1.  গ্রেডিয়েন্ট ভ্যানিশিং প্রবলেম।
2.  মডেল সাইজ বড় এবং ট্রেনিং ধীর।

**ResNet এর সমাধান:**

Residual Block গুলো ইনপুটকে সরাসরি আউটপুটের সাথে যোগ করে দেয়। নেটওয়ার্ক শেখে শুধুমাত্র "Residual" (অর্থাৎ পরিবর্তন) অংশটি।

---

## **Block 3: Core Concept - Residual Learning**

ধরা যাক, কোনো লেয়ারের ইনপুট হলো `x`। সাধারণ CNN-তে আমরা শিখতে চাই `H(x)` (মূল ফাংশন)।

ResNet এই ধারণাটি পাল্টে দেয়। এটি শেখে `F(x) = H(x) - x`। মানে, এটি রেসিডুয়াল (Residual) শেখে।

আউটপুট হয়: **`H(x) = F(x) + x`**

- `x` ইনপুটকে সরাসরি আউটপুটের সাথে **Skip Connection** এর মাধ্যমে যোগ করা হয়।
- নেটওয়ার্ক যদি দেখে যে `x` ই যথেষ্ট (অর্থাৎ নতুন কোনো ফিচার শেখার দরকার নেই), তাহলে `F(x)`-কে জিরো রেখে `H(x)=x` করে দেয়। এটা Gradient vanishing ঠেকায়।

---

# **Block 4: Key Components of ResNet-50**

ResNet-50 কে প্রধানত দুটি ব্লকে ভাগ করা যায়:

### A. Identity Block
- যখন ইনপুট ও আউটপুটের ডাইমেনশন একই থাকে।
- শুধু স্ট্যান্ডার্ড কনভোলুশন (Conv), ব্যাচ নরমালাইজেশন (BatchNorm), এবং রিলু (ReLU) ব্যবহার করা হয়।
- ইনপুট `x` কে আউটপুট `F(x)` এর সাথে সরাসরি যোগ করে দেওয়া হয়।

### B. Convolutional Block
- যখন ইনপুট ও আউটপুটের ডাইমেনশন ভিন্ন হয় (বিশেষ করে Image Downsampling করার সময়)।
- Skip Connection-এ একটি **1x1 Convolution** যোগ করা হয়।
- এর কাজ হলো:
    - ডাইমেনশন ম্যাচ করানো।
    - ফিল্টার সংখ্যা (Channels) বাড়ানো বা কমানো।

---


## **Block 5: Architecture Layout of ResNet-50 (Layer by Layer)**

ResNet-50 এ ৫টি প্রধান Stage আছে। **৩০০x৩০০x৩** ইমেজ (300x300 pixels, 3 color channels) দিয়ে শুরু করা যাক।

**Step-by-Step ডাইমেনশন ক্যালকুলেশন:**

| Stage | Layer Description | Output Shape |
| :--- | :--- | :--- |
| **Stage 0 (Input Stem)** | `Conv2D (7x7, stride 2)` + `MaxPool (3x3, stride 2)` | **75x75x64** |
| **Stage 1** | Conv Block (64) + 2 Identity Blocks | **75x75x256** |
| **Stage 2** | Conv Block (Stride 2) + 3 Identity Blocks | **38x38x512** |
| **Stage 3** | Conv Block (Stride 2) + 5 Identity Blocks | **19x19x1024** |
| **Stage 4** | Conv Block (Stride 2) + 2 Identity Blocks | **10x10x2048** |
| **Final Stage** | Global Average Pooling + FC (1000) + Softmax | **1000** (Class Scores) |

> **উদাহরণ সারাংশ**: 300x300 ইমেজ ধীরে ধীরে সঙ্কুচিত হয়ে 10x10 হয়, কিন্তু নেটওয়ার্কের Depth (Channels) 3 থেকে বেড়ে সর্বোচ্চ 2048 পর্যন্ত যায়।

---

# **Block 6: How ResNet-50 Solves Vanishing Gradient**

Backpropagation এর সময় গ্রেডিয়েন্ট ধীরে ধীরে ছোট হয়ে যায়। ResNet-50 তে, Backpropagation করার সময় গ্রেডিয়েন্ট দুটি পথ পায়:

1.  **Main Path**: কনভোলুশনের মধ্য দিয়ে অল্প গ্রেডিয়েন্ট আসে।
2.  **Shortcut Path (Skip Connection)**: `x` ইনপুট থেকে সরাসরি `H(x)` এ গ্রেডিয়েন্ট যায়। (Derivative 1)

**গণিত:** যদি আউটপুট `H(x) = F(x) + x` হয়, তাহলে গ্রেডিয়েন্ট হয়:

`dH/dx = dF/dx + 1`

এই `+1` অংশটি নিশ্চিত করে যে গভীর স্তরেও গ্রেডিয়েন্ট পুরোপুরি বিলুপ্ত হয়ে যাবে না। Skip Connection গভীর নেটওয়ার্ককে "Identity" ফাংশন শিখতে বাধ্য করে, যা দ্রুত কনভার্জ করে।

---


# **Block 7: Backpropagation Update Mechanism**

ধরা যাক, ResNet-50 একটি ছবি দেখে ভুলবশত কুকুরের জায়গায় বিড়াল ধরে ফেলল।

1.  **Forward Pass**: ছবিটি যত লেয়ার পাড়ি দেয়, আউটপুটে `Loss` (ক্ষতি) বের হয়।
2.  **Gradient Flow**: Backpropagation চলাকালে, যখন পিছনের দিকে যাওয়া হয়:
    - **Without ResNet (Plain)**: গভীর লেয়ারে পৌঁছালে গ্রেডিয়েন্ট `0.00000xxx` হয়ে যায়। (একদম ছোট)
    - **With ResNet**: Skip Connection কারণে গ্রেডিয়েন্ট `"Some Small Value + 1"` হয়। **`1`** এর মানটা গ্রেডিয়েন্টকে "Highway" (হাইওয়ে) প্রদান করে, ফলে গ্রেডিয়েন্ট নষ্ট হয় না।
3.  **Weight Update**: এই ভালো গ্রেডিয়েন্ট ব্যবহার করে Weight গুলো আপডেট হয়। আগের লেয়ারগুলো বুঝতে পারে "বিড়াল" চেনার জন্য ঠিকঠাক ফিচার বানাতে হবে কি করে।
4.  **Result**: পরবর্তী Iteration-এ মডেলটি সঠিকভাবে কুকুর শনাক্ত করতে শিখে যায়।

---

# **Block 8: Summary**

- **Architecture**: 50 Layer deep, কিন্তু ট্রেনিং সহজ।
- **Key Feature**: **Identity Shortcut Connections** (এড়িয়ে যাওয়ার পথ)।
- **Block Types**:
    - *Identity Block* (ডাইমেনশন অপরিবর্তিত)
    - *Convolutional Block* (ডাইমেনশন পরিবর্তিত)
- **Problem Solved**: Vanishing Gradient এর অভিশাপ দূর করেছে, ফলে খুব গভীর নেটওয়ার্ক ট্রেন করা সম্ভব হয়েছিল।